<a href="https://colab.research.google.com/github/Razan445/IT326-Project/blob/main/Phase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Razan445/IT326-Project/refs/heads/main/Dataset/Raw_dataset.csv")


## Data Preprocessing – Technique 1: Cleaning & Noise Removal



In this section, missing values were handled and outliers were treated so the dataset is clean and ready for the next preprocessing steps.




In [ ]:
# ===== 1. Snapshot of the raw dataset =====

print("Shape:", df.shape)

display(df.head(10))



# ===== 2. Work on a copy (raw stays untouched) =====

df_clean = df.copy()

print("\nMissing values before:\n", df_clean.isnull().sum())

print("\nDuplicates:", df_clean.duplicated().sum())



# ===== 3. Handle missing values =====

df_clean["Age"] = df_clean.groupby(["Sex", "Pclass"])["Age"].transform(lambda x: x.fillna(x.median()))



df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])

df_clean = df_clean.drop(columns=["Cabin"])  # ~77% missing

print("\nMissing values after:", df_clean.isnull().sum().sum())



# ===== 4. Fare outliers (IQR + capping) =====

q1, q3 = df_clean["Fare"].quantile([0.25, 0.75])

max_limit = q3 + 1.5 * (q3 - q1)

print(f"\nFare outliers: {(df_clean['Fare'] > max_limit).sum()}")

print(f"Upper limit: {max_limit:.2f}")

print("Max Fare before:", df_clean["Fare"].max())

df_clean["Fare"] = df_clean["Fare"].clip(upper=max_limit)

print("Max Fare after:", df_clean["Fare"].max())



# ===== 5. Before / After summary =====

print("\nRows before:", df.shape[0], "| after:", df_clean.shape[0])

print("Columns before:", df.shape[1], "| after:", df_clean.shape[1])

display(df_clean.head(10))



# ===== 6. Save for Student 4 =====

df_clean.to_csv("cleaned_step1.csv", index=False)





Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



Missing values before:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicates: 0

Missing values after: 0

Fare outliers: 116
Upper limit: 65.63
Max Fare before: 512.3292
Max Fare after: 65.6344

Rows before: 891 | after: 891
Columns before: 12 | after: 11


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,65.6344,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
5,6,0,3,"Moran, Mr. James",male,25.0,0,0,330877,8.4583,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,C


### Justification



**Why:** The raw dataset contained missing values in Age (177), Cabin (687) and Embarked (2). Decision Tree and K-means in scikit-learn cannot handle missing values. The Fare column also had 116 outliers (max = 512.33) that distort distance-based clustering such as K-means.



**How:**

- **Age:** filled with the median grouped by Sex and Pclass, because age differs across these groups, which makes it more accurate than a global median.

- **Embarked:** filled with the mode since only 2 values were missing.

- **Cabin:** dropped because about 77% of its values were missing, and imputing would add noise.

- **Fare:** outliers were detected using the IQR rule and capped at Q3 + 1.5×IQR (= 65.63) instead of deleting rows, to keep all records.



**Columns affected:** Age, Cabin, Embarked, Fare



**Result:** Missing values dropped from 866 to 0. All 891 rows were preserved. The maximum Fare decreased from 512.33 to 65.63, giving a less skewed distribution that is ready for the next preprocessing steps.




